# NII Scenarios 2025–2026 — Greek Systemic Banks

Scenario model built on FY2022–FY2024 data, using the rate/volume framework of `03_analysis/04_nii_rate_volume_walk.ipynb`. The Streamlit app (Forecast page) uses the same method (`05_streamlit_app/utils/data.py`, `forecast_nii`).

**Model:** `NII = NIM × average total assets`. NIM and assets are projected separately and multiplied.

**ECB deposit rate scenarios (yearly average)**
| Scenario | 2025 | 2026 | Assumption |
|----------|------|------|------------|
| Base | 2.25% | 2.00% | Gradual cuts to about 2% |
| Hawkish | 3.00% | 2.75% | Inflation returns, fewer cuts |
| Dovish | 1.75% | 1.50% | Faster cuts, weaker growth |

The scenarios are yearly averages because the NIM sensitivity is estimated on yearly average rates. For reference, the actual ECB deposit rate reached 2.00% in June 2025. FY2025 results have since been published; a back-test against them is the natural next step.

**NIM:** 2024 NIM + the bank's NIM sensitivity × (scenario rate − 2024 average rate). The sensitivity is the slope of a per-bank regression of NIM on the yearly average ECB deposit rate, 2022–2024. Three years of data, with 2022 at near-zero rates, give a steep slope, while hedging and lower deposit costs soften the real effect, so treat the lower paths with caution.

**Assets:** total assets grow 2% a year from the end-2024 balance sheet, in every scenario, so the scenarios differ only in rates. Starting from end-2024 captures Eurobank's consolidation of Hellenic Bank in mid-2024.

> **Disclaimer:** an illustration of method, not investment advice. Three yearly points cannot support real statistical inference, so the bands below are illustrative ranges, not probabilities.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} bank-year rows.')
print('Historical NIM and Total Assets:')
print(kpis[['bank','year','nii','nim','total_assets']].to_string(index=False))

Loaded 12 bank-year rows.
Historical NIM and Total Assets:
        bank  year    nii  nim  total_assets
  Alpha Bank  2022 1173.8 1.55       78011.4
  Alpha Bank  2023 1659.0 2.21       72421.0
  Alpha Bank  2024 1645.0 2.28       72075.0
    Eurobank  2022 1480.0 1.86       81474.0
    Eurobank  2023 2174.0 2.70       79781.0
    Eurobank  2024 2507.0 2.77      101150.0
         NBG  2022 1369.0 1.69       78113.0
         NBG  2023 2263.0 2.96       74584.0
         NBG  2024 2356.0 3.15       74957.0
Piraeus Bank  2022 1353.0 1.75       74645.0
Piraeus Bank  2023 2003.0 2.65       76450.0
Piraeus Bank  2024 2088.0 2.67       80044.0


In [2]:
# ── ECB deposit rate history and scenarios ────────────────────────────────────
ecb = pd.read_csv(DATA_DIR.parent / 'manual' / 'ecb_deposit_rate.csv')
ECB_HIST = dict(zip(ecb['year'], ecb['dfr_avg']))   # yearly averages

ECB_SCENARIOS = {                                    # yearly averages
    'Base':     {2025: 2.25, 2026: 2.00},
    'Hawkish':  {2025: 3.00, 2026: 2.75},
    'Dovish':   {2025: 1.75, 2026: 1.50},
}
SCENARIO_COLORS = {'Base': '#FFFFFF', 'Hawkish': '#EF553B', 'Dovish': '#636EFA'}

ecb_hist_vals = np.array([ECB_HIST[y] for y in [2022, 2023, 2024]])

bank_params = {}
for bank in BANKS:
    nim_vals = kpis[kpis['bank'] == bank].sort_values('year')['nim'].values   # % of average assets
    slope, intercept, r, p, se = stats.linregress(ecb_hist_vals, nim_vals)
    residuals = nim_vals - (intercept + slope * ecb_hist_vals)
    rmse = np.std(residuals, ddof=1)
    bank_params[bank] = {'slope': slope, 'rmse': rmse, 'r2': r**2, 'nim_2024': nim_vals[-1]}
    print(f'{bank}: NIM sensitivity {slope:.3f}pp per 1pp of ECB rate | R²={r**2:.3f} | RMSE={rmse:.3f}pp')

print('\nNote: three data points per bank, so the fit is directional only.')

Eurobank: NIM sensitivity 0.253pp per 1pp of ECB rate | R²=0.999 | RMSE=0.018pp
Alpha Bank: NIM sensitivity 0.202pp per 1pp of ECB rate | R²=1.000 | RMSE=0.007pp
Piraeus Bank: NIM sensitivity 0.262pp per 1pp of ECB rate | R²=0.993 | RMSE=0.045pp
NBG: NIM sensitivity 0.397pp per 1pp of ECB rate | R²=1.000 | RMSE=0.012pp

Note: three data points per bank, so the fit is directional only.


In [3]:
# ── Average assets: 2% a year from the end-2024 balance sheet ─────────────────
FORECAST_YEARS = [2025, 2026]
ASSET_GROWTH = 0.02

k24 = kpis[kpis['year'] == 2024].set_index('bank')
asset_forecasts = {}
for bank in BANKS:
    ta = k24.loc[bank, 'total_assets']
    asset_forecasts[bank] = {
        2025: ta * (1 + ASSET_GROWTH / 2),                        # average of end-2024 and end-2025
        2026: ta * (1 + ASSET_GROWTH) * (1 + ASSET_GROWTH / 2),   # average of end-2025 and end-2026
    }
    print(f"{bank}: end-2024 assets {ta:,.0f} | 2024 average {k24.loc[bank, 'avg_total_assets']:,.0f} | "
          f"2025F average {asset_forecasts[bank][2025]:,.0f} | 2026F average {asset_forecasts[bank][2026]:,.0f}")

print('\nNote: Eurobank consolidated Hellenic Bank from 30 June 2024, so its end-2024 assets are well above its 2024 average.')

Eurobank: end-2024 assets 101,150 | 2024 average 90,466 | 2025F average 102,162 | 2026F average 104,205
Alpha Bank: end-2024 assets 72,075 | 2024 average 72,248 | 2025F average 72,796 | 2026F average 74,252
Piraeus Bank: end-2024 assets 80,044 | 2024 average 78,247 | 2025F average 80,844 | 2026F average 82,461
NBG: end-2024 assets 74,957 | 2024 average 74,770 | 2025F average 75,707 | 2026F average 77,221

Note: Eurobank consolidated Hellenic Bank from 30 June 2024, so its end-2024 assets are well above its 2024 average.


In [4]:
# ── Build NII forecasts per bank × scenario × year ───────────────────────────
Z_80 = 1.282  # z-score for 80% CI
Z_95 = 1.960  # z-score for 95% CI

forecasts = []
for bank in BANKS:
    p = bank_params[bank]
    for scenario, ecb_path in ECB_SCENARIOS.items():
        for yr in FORECAST_YEARS:
            nim_f    = p['nim_2024'] + p['slope'] * (ecb_path[yr] - ECB_HIST[2024])  # NIM forecast (%)
            assets_f = asset_forecasts[bank][yr]
            nii_f    = nim_f / 100 * assets_f  # NII = NIM × Assets
            # Illustrative band: NIM regression RMSE carried through NIM × Assets
            nii_se   = p['rmse'] / 100 * assets_f
            forecasts.append({
                'bank': bank, 'scenario': scenario, 'year': yr,
                'nim_f': nim_f, 'assets_f': assets_f,
                'nii_f': nii_f,
                'nii_lo80': nii_f - Z_80 * nii_se,
                'nii_hi80': nii_f + Z_80 * nii_se,
                'nii_lo95': nii_f - Z_95 * nii_se,
                'nii_hi95': nii_f + Z_95 * nii_se,
            })

fcst = pd.DataFrame(forecasts)

print('\n── Base Scenario NII Forecast ──')
base = fcst[fcst['scenario']=='Base'][['bank','year','nim_f','nii_f','nii_lo80','nii_hi80']].round(2)
print(base.to_string(index=False))


── Base Scenario NII Forecast ──
        bank  year  nim_f   nii_f  nii_lo80  nii_hi80
    Eurobank  2025   2.39 2446.75   2422.76   2470.73
    Eurobank  2026   2.33 2429.67   2405.21   2454.14
  Alpha Bank  2025   1.98 1442.50   1435.60   1449.39
  Alpha Bank  2026   1.93 1433.92   1426.88   1440.95
Piraeus Bank  2025   2.28 1844.90   1797.95   1891.85
Piraeus Bank  2026   2.22 1827.76   1779.87   1875.65
         NBG  2025   2.56 1939.47   1928.20   1950.73
         NBG  2026   2.46 1901.54   1890.05   1913.03


In [5]:
# ── Chart 1: Fan Chart — NII Forecast 2025-2026 per Bank ──────────────────────
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    vertical_spacing=0.16, horizontal_spacing=0.10,
)

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bkpis = kpis[kpis['bank'] == bank].sort_values('year')

    # Historical line
    fig1.add_trace(go.Scatter(
        x=bkpis['year'].tolist(), y=bkpis['nii'].tolist(),
        mode='lines+markers', line=dict(color=COLORS[bank], width=2.5),
        marker=dict(size=8), name=f'{bank} Historical',
        showlegend=(bi == 0), legendgroup='hist',
    ), row=row+1, col=col+1)

    # Scenario lines
    for scenario in ECB_SCENARIOS:
        sc_fcst = fcst[(fcst['bank']==bank) & (fcst['scenario']==scenario)].sort_values('year')
        # Connect from 2024 actual to forecast
        x_all = [2024] + sc_fcst['year'].tolist()
        y_all = [bkpis[bkpis['year']==2024]['nii'].values[0]] + sc_fcst['nii_f'].tolist()

        fig1.add_trace(go.Scatter(
            x=x_all, y=y_all,
            mode='lines+markers',
            line=dict(color=SCENARIO_COLORS[scenario], width=1.8, dash='dot'),
            marker=dict(size=7),
            name=f'{scenario} scenario',
            showlegend=(bi == 0),
            legendgroup=scenario,
        ), row=row+1, col=col+1)

        if scenario == 'Base':
            # CI bands
            x_fcst = sc_fcst['year'].tolist()
            fig1.add_trace(go.Scatter(
                x=x_fcst + x_fcst[::-1],
                y=sc_fcst['nii_hi80'].tolist() + sc_fcst['nii_lo80'].tolist()[::-1],
                fill='toself', fillcolor='rgba(255,255,255,0.12)',
                line=dict(color='rgba(255,255,255,0)'),
                name='80% CI (Base)', showlegend=(bi == 0), legendgroup='ci80',
            ), row=row+1, col=col+1)
            fig1.add_trace(go.Scatter(
                x=x_fcst + x_fcst[::-1],
                y=sc_fcst['nii_hi95'].tolist() + sc_fcst['nii_lo95'].tolist()[::-1],
                fill='toself', fillcolor='rgba(255,255,255,0.06)',
                line=dict(color='rgba(255,255,255,0)'),
                name='95% CI (Base)', showlegend=(bi == 0), legendgroup='ci95',
            ), row=row+1, col=col+1)

    # Vertical separator at 2024
    fig1.add_vline(x=2024.5, line_dash='dash', line_color='rgba(255,255,255,0.2)',
                   row=row+1, col=col+1)

fig1.update_layout(
    title_text='<b>NII Forecast 2025–2026</b> | Base / Hawkish / Dovish ECB scenarios | Shaded = 80%/95% CI (Base)',
    title_font_size=15, template='plotly_dark', height=620,
    legend=dict(orientation='h', y=-0.16, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.update_yaxes(title_text='NII (€ million)')
fig1.show()

In [6]:
# ── Chart 2: Sector NII by Scenario ───────────────────────────────────────────
sector_hist = kpis.groupby('year')['nii'].sum().reset_index()
sector_fcst = fcst.groupby(['scenario','year'])[['nii_f','nii_lo80','nii_hi80','nii_lo95','nii_hi95']].sum().reset_index()

fig2 = go.Figure()
# Historical
fig2.add_trace(go.Scatter(
    x=sector_hist['year'], y=sector_hist['nii'],
    mode='lines+markers', line=dict(color='white', width=3),
    marker=dict(size=10), name='Historical sector NII',
))

for scenario in ECB_SCENARIOS:
    sc = sector_fcst[sector_fcst['scenario']==scenario].sort_values('year')
    # Bridge from 2024 actual
    x_all = [2024] + sc['year'].tolist()
    y_all = [sector_hist[sector_hist['year']==2024]['nii'].values[0]] + sc['nii_f'].tolist()
    fig2.add_trace(go.Scatter(
        x=x_all, y=y_all,
        mode='lines+markers+text',
        line=dict(color=SCENARIO_COLORS[scenario], width=2.5, dash='dot'),
        marker=dict(size=9),
        text=[None] + [f'{v/1000:.1f}bn' for v in sc['nii_f'].tolist()],
        textposition='top center',
        name=f'{scenario} scenario',
    ))

fig2.add_vline(x=2024.5, line_dash='dash', line_color='rgba(255,255,255,0.25)')
fig2.add_annotation(x=2024.6, y=sector_hist['nii'].max()*0.9,
    text='Forecast →', showarrow=False, font=dict(color='white', size=12))

fig2.update_layout(
    title_text='<b>4-Bank Sector NII Forecast</b> | 2025–2026 by ECB scenario (€ million)',
    template='plotly_dark', height=400,
    yaxis_title='Sector NII (€ million)',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig2.show()

In [7]:
# ── Summary Table ─────────────────────────────────────────────────────────────
print('\n── NII Forecast Summary (€ million) ──')
pivot = fcst[fcst['year']==2026].pivot_table(index='bank', columns='scenario', values='nii_f').round(0)
pivot['Historical_2024'] = kpis[kpis['year']==2024].set_index('bank')['nii']
pivot = pivot[['Historical_2024', 'Dovish', 'Base', 'Hawkish']]
print('2026 NII Forecast vs 2024 Actual:')
print(pivot.to_string())

print('\nECB Rate Assumptions:')
for sc, path in ECB_SCENARIOS.items():
    print(f'  {sc}: 2025={path[2025]:.2f}%, 2026={path[2026]:.2f}%')


── NII Forecast Summary (€ million) ──
2026 NII Forecast vs 2024 Actual:
scenario      Historical_2024  Dovish    Base  Hawkish
bank                                                  
Alpha Bank             1645.0  1359.0  1434.0   1546.0
Eurobank               2507.0  2298.0  2430.0   2628.0
NBG                    2356.0  1748.0  1902.0   2132.0
Piraeus Bank           2088.0  1720.0  1828.0   1990.0

ECB Rate Assumptions:
  Base: 2025=2.25%, 2026=2.00%
  Hawkish: 2025=3.00%, 2026=2.75%
  Dovish: 2025=1.75%, 2026=1.50%


In [8]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(fcst) == len(BANKS) * len(ECB_SCENARIOS) * len(FORECAST_YEARS)
assert fcst['nii_f'].gt(0).all(), 'All NII forecasts must be positive'
assert (fcst['nii_hi80'] > fcst['nii_lo80']).all(), 'CI upper must exceed lower'
assert (fcst['nii_hi95'] > fcst['nii_hi80']).all(), '95% CI must be wider than 80%'

# Hawkish scenario should give higher NII than Dovish for all banks in 2026
hawk_nii = fcst[(fcst['scenario']=='Hawkish') & (fcst['year']==2026)].set_index('bank')['nii_f']
dove_nii = fcst[(fcst['scenario']=='Dovish')  & (fcst['year']==2026)].set_index('bank')['nii_f']
for bank in BANKS:
    if bank in hawk_nii.index and bank in dove_nii.index:
        assert hawk_nii[bank] > dove_nii[bank], f'{bank}: Hawkish NII should exceed Dovish'

print('All checks passed.')
base_sector_2026 = fcst[(fcst['scenario']=='Base') & (fcst['year']==2026)]['nii_f'].sum()
hist_sector_2024 = kpis[kpis['year']==2024]['nii'].sum()
print(f'   Base scenario sector NII 2026: {base_sector_2026:.0f}m (vs {hist_sector_2024:.0f}m in 2024)')
print(f'   Implied sector NII change: {(base_sector_2026/hist_sector_2024-1)*100:+.1f}%')

All checks passed.
   Base scenario sector NII 2026: 7593m (vs 8596m in 2024)
   Implied sector NII change: -11.7%
